# Contextual Zooming Bandit — Mercado Eléctrico Alemán

**Trabajo de Fin de Grado · Universidad de Málaga**

Implementación del algoritmo *Contextual Zooming* de Singhal et al. (2025) aplicado a la optimización de pujas de un parque eólico en el mercado eléctrico alemán. El notebook reproduce el experimento numérico del paper sobre datos reales del período **julio 2022 – marzo 2024** obtenidos de la plataforma [SMARD](https://www.smard.de/).

---

## 1. Contexto del problema

Un productor eólico con ~20 GW instalados en la zona de control de 50Hertz (noreste de Alemania) debe decidir cada hora cuánta energía ofertar en el mercado *day-ahead*. La decisión óptima depende de las condiciones del mercado, que el productor solo conoce de forma ruidosa antes de la subasta.

El problema se formula como un **bandido contextual con similitud** en el espacio producto $\mathcal{P} = [0,1]^4$:

| Componente | Descripción |
|---|---|
| **Brazo** $y_t \in [0,1]$ | Puja normalizada; $y=0.5$ equivale a ofertar la previsión $\hat{g}^w_t$ |
| **Contexto** $x_t \in [0,1]^3$ | $(\hat{\gamma}_t, \hat{\lambda}^I_t, \hat{\eta}^I_t)$ — señales de mercado normalizadas |
| **Recompensa** $\rho_t \in [-1,1]$ | Mejora de ingreso respecto al *Forecast Bid*, normalizada |

El algoritmo recibe las recompensas con un **retardo de $W=24$ rondas** (las 24 horas del día se liquidan juntas al final de la jornada).

## 2. Modelo de mercado

El ingreso del productor en la hora $t$ es:

$$I_t = \underbrace{(\lambda^S_t + \eta^S_t \cdot \Delta_t) \cdot p^w_t}_{\text{mercado diario}} + \underbrace{(\lambda^I_t + \eta^I_t \cdot \Delta_t) \cdot (g^w_{\text{real},t} - p^w_t)}_{\text{mercado de desvíos}}$$

donde $\Delta_t = (y_t - 0.5) \times 2 \times 250$ MWh es la desviación respecto a la previsión. Las sensibilidades $\eta^S_t < 0$ y $\eta^I_t < 0$ capturan el efecto *price-maker*: ofertar más deprime ambos precios.

In [ ]:
# Importaciones y configuración
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['figure.dpi'] = 120

print("Librerías cargadas correctamente.")

In [ ]:
# Cargar módulos del algoritmo
from bola       import Bola, calcular_indice, DIM_CTX, RADIO_MIN
from mercado    import calcular_ingreso, calcular_rango, Y_FC, DELTA_MW
from bandit     import (cargar_datos as cargar_bandit,
                        simular_bandit,
                        calcular_resultados as calc_res_bandit,
                        guardar_csv as guardar_bandit)
from benchmarks import (cargar_datos as cargar_bench,
                        simular_oraculo, simular_d1, simular_forecast,
                        calcular_resultados as calc_res_bench,
                        guardar_csv as guardar_bench)

print("Módulos cargados.")

## 3. Datos

El *dataset* contiene **15 360 horas** (julio 2022 – marzo 2024) con las siguientes variables:

- `lambda_S`, `lambda_I` — precios *day-ahead* y de desvío (€/MWh)  
- `eta_S`, `eta_I` — sensibilidades de precio  
- `g_w_hat_MWh`, `g_w_real_MWh` — previsión y generación real eólica (MWh)

Los contextos en `contextos.csv` están normalizados a $[0,1]$ con ruido $t$-Student de escala 0.05.

In [ ]:
# Cargar y explorar los datos
df_mercado, ctx, T, fechas = cargar_bandit()

print(f"\nPequeño vistazo al dataset:")
display(pd.read_csv('dataset_final.csv', parse_dates=['timestamp']).head(3))

In [ ]:
# Estadísticas descriptivas de las variables de mercado
df = pd.read_csv('dataset_final.csv', parse_dates=['timestamp']).set_index('timestamp')
stats = df[['lambda_S','lambda_I','g_w_hat_MWh','g_w_real_MWh']].describe().round(1)
stats.index = ['count','media','std','min','25%','50%','75%','max']
print("Estadísticas descriptivas:")
display(stats)

In [ ]:
# Visualizar series temporales de precios
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(df.index, df['lambda_S'], lw=0.6, color='#1f77b4', alpha=0.8, label='Precio day-ahead (λˢ)')
axes[0].plot(df.index, df['lambda_I'], lw=0.6, color='#d62728', alpha=0.8, label='Precio desvío (λᴵ)')
axes[0].set_ylabel('€/MWh')
axes[0].set_title('Precios de mercado — jul 2022 / mar 2024')
axes[0].legend(fontsize=9)
axes[0].grid(True, ls=':', alpha=0.4)

axes[1].plot(df.index, df['g_w_hat_MWh'], lw=0.6, color='#2ca02c', alpha=0.8, label='Previsión (ĝʷ)')
axes[1].plot(df.index, df['g_w_real_MWh'], lw=0.6, color='#ff7f0e', alpha=0.5, label='Real (gʷ)')
axes[1].set_ylabel('MWh')
axes[1].set_title('Generación eólica')
axes[1].legend(fontsize=9)
axes[1].grid(True, ls=':', alpha=0.4)

axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 4. El algoritmo Contextual Zooming

El *Contextual Zooming* mantiene un conjunto de bolas $\mathcal{B}_t \subset [0,1]^4$ que particionan progresivamente el espacio contexto-oferta. Cada bola $B(c, r)$ tiene un **índice UCB**:

$$\mathcal{I}(B) = r(B) + \min_{B' \in \mathcal{B}} \left[ \nu(B') + r(B') + \text{conf}(B') + D(B, B') \right]$$

donde $\nu(B)$ es la recompensa media, $\text{conf}(B) = \sqrt{\log T / (1+n_B)}$ el intervalo de confianza y $D$ la distancia entre centros.

En cada batch diario:
1. **Predict**: para cada hora $t$, seleccionar la bola con mayor índice y muestrear una puja dentro de ella  
2. **Observe**: recibir las 24 recompensas al final del día  
3. **Update**: actualizar contadores y activar bolas hijas donde la certeza lo justifique

In [ ]:
# Inspeccionar la clase Bola
from bola import Bola, calcular_indice
import numpy as np

Bola._cnt = 0
np.random.seed(42)

b0 = Bola([0.5, 0.5, 0.5, 0.5], 1.0)   # bola inicial
b0.n, b0.reward = 50, 12.5

b1 = Bola([0.2, 0.6, 0.4, 0.3], 0.5)   # bola hija

print(f"Bola inicial (50 obs):  ν={b0.nu():.4f}  conf={b0.conf():.4f}  I={calcular_indice(b0,[b0,b1]):.4f}")
print(f"Bola hija   (0 obs):   ν={b1.nu():.4f}  conf={b1.conf():.4f}  I={calcular_indice(b1,[b0,b1]):.4f}")
print(f"\nLa bola hija tiene mayor índice → el algoritmo la prefiere para explorar")

## 5. Simulación completa

A continuación se ejecutan las cuatro estrategias evaluadas:

| Estrategia | Descripción |
|---|---|
| **Oráculo** | Puja óptima con datos reales de la hora $t$ — cota superior teórica |
| **Bandido Contextual** | Contextual Zooming con $W=24$, $d_c=4$ |
| **Predicción D-1** | Optimiza con datos de $t-24$ y aplica la puja hoy |
| **Forecast Bid** | Siempre puja $\hat{g}^w_t$ ($\Delta_t=0$) — referencia base |

> ⏱️ **La simulación completa tarda varios minutos.** El bandit procesa 15 360 horas en 640 batches; los benchmarks requieren grid search en cada hora.

In [ ]:
# ── Bandit ────────────────────────────────────────────────────
np.random.seed(42)
print("[1/4] Contextual Zooming...")
pujas_cz, rew_cz, n_bolas_hist, bolas_fin = simular_bandit(ctx, df_mercado, T)
ing_cz, da_cz, rt_cz, ing_fc_b, deltas_cz = calc_res_bandit(pujas_cz, df_mercado, T)
print(f"✓ Bolas finales: {len(bolas_fin)}")

In [ ]:
# ── Benchmarks ───────────────────────────────────────────────
df_bench, T_b, fechas_b = cargar_bench()

print("[2/4] Oráculo...")
p_or = simular_oraculo(df_bench, T_b)

print("\n[3/4] Predicción D-1...")
p_d1 = simular_d1(df_bench, T_b)

print("\n[4/4] Forecast Bid...")
p_fc = simular_forecast(T_b)

print("\nCalculando ingresos...")
ing_or,  da_or,  rt_or,  delta_or  = calc_res_bench(p_or, df_bench, T_b)
ing_d1,  da_d1,  rt_d1,  delta_d1  = calc_res_bench(p_d1, df_bench, T_b)
ing_fc2, da_fc,  rt_fc,  _         = calc_res_bench(p_fc, df_bench, T_b)

T_min  = min(T, T_b)
ing_fc = ing_fc2[:T_min]
print("✓ Benchmarks completados.")

## 6. Resultados

In [ ]:
# Tabla de resultados
resultados = {
    'Estrategia':   ['Oráculo †', 'Bandido Contextual', 'Predicción D-1', 'Forecast Bid'],
    'DA (€/h)':     [f"{da_or.mean():,.0f}", f"{da_cz[:T_min].mean():,.0f}",
                     f"{da_d1.mean():,.0f}", f"{da_fc.mean():,.0f}"],
    'RT (€/h)':     [f"{rt_or.mean():+,.0f}", f"{rt_cz[:T_min].mean():+,.0f}",
                     f"{rt_d1.mean():+,.0f}", f"{rt_fc.mean():+,.0f}"],
    'Total (€/h)':  [f"{ing_or.mean():,.0f}", f"{ing_cz[:T_min].mean():,.0f}",
                     f"{ing_d1.mean():,.0f}", f"{ing_fc2.mean():,.0f}"],
    'vs FC (€/h)':  [f"{(ing_or.mean()-ing_fc2.mean()):+,.0f}",
                     f"{(ing_cz[:T_min].mean()-ing_fc2.mean()):+,.0f}",
                     f"{(ing_d1.mean()-ing_fc2.mean()):+,.0f}", "—"],
}
df_res = pd.DataFrame(resultados)
df_res = df_res.set_index('Estrategia')
print("† Cota superior teórica; no es una estrategia causal.\n")
display(df_res)

## 7. Figuras

### 7.1 Arrepentimiento medio acumulado vs cota teórica

In [ ]:
D_C    = DIM_CTX + 1
W_lote = 24

fechas_t = fechas[:T_min]
ing_cz_t = ing_cz[:T_min]
ing_or_t = ing_or[:T_min]
ing_fc_t = ing_fc[:T_min]
ing_d1_t = ing_d1[:T_min]

def regret_medio(ing, ing_ref):
    n = min(len(ing), len(ing_ref))
    return np.cumsum(ing_ref[:n] - ing[:n]) / np.arange(1, n+1)

reg_cz = regret_medio(ing_cz_t, ing_or_t)

t_arr  = np.arange(1, T_min+1)
e1     = (D_C+1) / (D_C+2)
e2     = (D_C-1) / (D_C+2)
forma  = (t_arr**e1 * np.log(np.maximum(t_arr,2)) + W_lote*t_arr**e2) / t_arr
burnin = max(1, int(0.05*T_min))
C      = (reg_cz[burnin:] / np.maximum(forma[burnin:], 1e-12)).max() * 1.05
reg_teo = C * forma

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(fechas_t, reg_cz,  color='#1f77b4', lw=2.0, label='Arrepentimiento empírico (Bandido)')
ax.plot(fechas_t, reg_teo, color='darkorange', lw=2.0, ls='--',
        label=f'Cota teórica Teorema 1 ($d_c={D_C}$, $W={W_lote}$)')
ax.axhline(0, color='black', lw=0.8, ls=':')
ax.set_ylabel('$R(t)/t$ (€)', fontsize=12)
ax.set_title('Fig. 6.1 — Arrepentimiento medio acumulado vs cota teórica', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, ls=':', alpha=0.4)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

### 7.2 Ingreso acumulado relativo al Forecast Bid

In [ ]:
ir_cz = np.cumsum(ing_cz_t - ing_fc_t)
ir_d1 = np.cumsum(ing_d1_t - ing_fc_t)
ir_or = np.cumsum(ing_or_t - ing_fc_t)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(fechas_t, ir_or/1e8, color='#2ca02c', lw=1.8, ls='--', label='Oráculo')
ax.plot(fechas_t, ir_cz/1e8, color='#1f77b4', lw=2.2, label='Bandido (CZ)')
ax.plot(fechas_t, ir_d1/1e8, color='#ff7f0e', lw=1.8, ls='-.', label='Predicción D-1')
ax.axhline(0, color='#d62728', lw=1.5, ls='--', label='Forecast Bid (ref.)')
ax.fill_between(fechas_t, ir_cz/1e8, 0, where=(ir_cz>0), alpha=0.12, color='#1f77b4')
ax.set_ylabel('Ingreso acumulado relativo (×10⁸ €)', fontsize=12)
ax.set_title('Fig. 6.2 — Ingreso acumulado relativo al Forecast Bid', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, ls=':', alpha=0.4)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

### 7.3 Desglose day-ahead / real-time

In [ ]:
ests = ['Oráculo', 'Bandido (CZ)', 'Predicción D-1', 'Forecast Bid']
idas = [da_or.mean(), da_cz[:T_min].mean(), da_d1.mean(), da_fc.mean()]
irts = [rt_or.mean(), rt_cz[:T_min].mean(), rt_d1.mean(), rt_fc.mean()]
cols = ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728']

x  = np.arange(len(ests))
bw = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(x+bw/2, idas, bw, color=cols, alpha=0.85, label='Mercado diario')
ax.barh(x-bw/2, irts, bw, color=cols, alpha=0.45, hatch='///', label='Mercado de desvíos')
ax.set_yticks(x)
ax.set_yticklabels(ests, fontsize=11)
ax.set_xlabel('Ingreso medio por hora (€)', fontsize=12)
ax.set_title('Fig. 6.3 — Desglose de ingresos: mercado diario vs mercado de desvíos', fontsize=13)
ax.legend(fontsize=10)
ax.axvline(0, color='black', lw=0.8)
ax.grid(True, axis='x', ls=':', alpha=0.4)
plt.tight_layout()
plt.show()

### 7.4 Distribución de la desviación de oferta $\Delta_t$

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
n, bins, patches_hist = ax.hist(deltas_cz, bins=50, alpha=0.75, edgecolor='white',
                                 label='Distribución de $\\Delta_t$')
for patch, left in zip(patches_hist, bins[:-1]):
    patch.set_facecolor('#1f77b4' if left < 0 else '#E87722')

ax.axvline(0, color='black', lw=1.5, ls='--', label='Forecast Bid ($\\Delta_t=0$)')

pct_neg = (deltas_cz < 0).mean() * 100
pct_pos = (deltas_cz > 0).mean() * 100
ymax = ax.get_ylim()[1]
ax.text(-180, ymax*0.85, f'$\\Delta_t < 0$\n{pct_neg:.1f}%\n(subpuja)',
        ha='center', fontsize=10, color='#1f77b4',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#1f77b4', alpha=0.8))
ax.text(+180, ymax*0.85, f'$\\Delta_t > 0$\n{pct_pos:.1f}%\n(sobrepuja)',
        ha='center', fontsize=10, color='#E87722',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#E87722', alpha=0.8))

ax.set_xlabel('Desviación de oferta $\\Delta_t$ (MWh)', fontsize=12)
ax.set_ylabel('Número de horas', fontsize=12)
ax.set_title('Fig. 6.4 — Distribución de la desviación de oferta del Bandido', fontsize=13)
ax.set_xlim(-260, 260)
ax.grid(True, ls=':', alpha=0.4)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### 7.5 Evolución del número de bolas activas

In [ ]:
fechas_lotes = fechas[::24][:len(n_bolas_hist)]
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(fechas_lotes, n_bolas_hist, color='#1f77b4', lw=1.8)
ax.fill_between(fechas_lotes, n_bolas_hist, alpha=0.15, color='#1f77b4')
ax.set_ylabel('Bolas activas $|\\mathcal{B}_t|$', fontsize=12)
ax.set_title('Fig. 6.5 — Evolución del número de bolas activas', fontsize=13)
ax.grid(True, ls=':', alpha=0.4)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()
print(f"Bolas finales: {len(bolas_fin)}")

## 8. Guardar resultados

Las celdas siguientes guardan los resultados en CSV para análisis posterior.

In [ ]:
guardar_bandit(pujas_cz, ing_cz, da_cz, rt_cz,
               ing_fc, rew_cz, deltas_cz,
               n_bolas_hist, df_mercado, fechas, T)

guardar_bench(p_or, p_d1, p_fc,
              ing_or, da_or, rt_or,
              ing_d1, da_d1, rt_d1,
              ing_fc2, da_fc, rt_fc,
              delta_or, delta_d1,
              df_bench, fechas_b, T_b)

print("✓ Resultados guardados en resultado_bandit.csv y resultado_benchmark.csv")

---

## Referencia

Singhal, A. et al. (2025). *Contextual Zooming Bandits for Wind Power Producer Bidding Strategy*. IEEE Transactions on Energy Markets, Policy and Regulation.